# 04 - Final Preprocessing & TensorFlow tf.data Pipeline
## Tai Phake Visual Speech Recognition (VSR) Project

**Objective**: Implement the complete, deterministic preprocessing and high-throughput data loading pipeline using **TensorFlow/Keras** across the canonical Tai Phake digits dataset:
- **30 Speakers**: `s1` through `s30`
- **11 Digits**: `d0` through `d10`
- **330 Total Utterances** (Canonical dataset)

### Pipeline Architecture:
1. **Imports & Reproducibility**: Seed initialization (`SEED = 42`) across Python, NumPy, and TensorFlow.
2. **Centralized Configuration**: Modular, parameter-driven configuration for crop dimensions ($96\times 96$), temporal sequence length ($T=30$), batch size ($B=16$), and flags.
3. **Dataset Discovery & Health Audit**: Discovery of all 330 utterances from `data/cropped_lips_dataset/`, numerical frame sorting, and gap/continuity validation.
4. **Class Mapping & Manifest Generation**: Deterministic class dictionary (`d0`–`d10` $\rightarrow$ 0–10) and export of `preprocessing_manifest.csv`.
5. **Deterministic Speaker-Independent Split**: 24 Train, 3 Val, 3 Test with zero speaker overlap, automated leakage assertion, and `speaker_split.json` export.
6. **Deterministic Temporal Normalization**: Strict normalization to exactly $T=30$ frames via uniform downsampling (>30) or deterministic interpolation (<30).
7. **TensorFlow Native Frame Loading**: `tf.io.read_file` and `tf.image.decode_jpeg` scaling to float32 $[0, 1]$.
8. **Temporally-Consistent Augmentation**: Sequence-level photometric and spatial transforms preserving inter-frame temporal coherence.
9. **High-Throughput tf.data Pipeline**: RAM-cached datasets with training shuffle, batching, and `tf.data.AUTOTUNE` prefetching.
10. **Validation & Batch Shape Verification**: Full dataset audit, class balance verification, and batch inspection `(B=16, T=30, H=96, W=96, C=3)`.
11. **Configuration Persistence**: Complete JSON config export to `results/dlib/preprocessing_config.json`.
12. **Model Input Interface Verification**: Demonstration of tensor forward pass through a Keras `TimeDistributed(CNN)` architecture.

## 1. Imports & Environment Setup
We initialize core computer vision, scientific computing, and deep learning libraries, and fix global seeds for end-to-end reproducibility.

In [1]:
# Section 1: Imports & Global Seeds
import os
import sys
import re
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import tensorflow as tf

# Seed for complete reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Python Version:     {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"OpenCV Version:     {cv2.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"Pandas Version:     {pd.__version__}")

/Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python Version:     3.12.13
TensorFlow Version: 2.21.0
OpenCV Version:     5.0.0
NumPy Version:      2.5.3
Pandas Version:     3.0.6


## 2. Centralized Pipeline Configuration
All hyperparameters, paths, and control flags are configured here to avoid hardcoded values across the pipeline.

In [ ]:
# Section 2: Configuration
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Paths
INPUT_CROPS_ROOT = PROJECT_ROOT / "data" / "cropped_lips_dataset"
RESULTS_DIR = PROJECT_ROOT / "results" / "dlib"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Input Lip Image Dimensions (Final Dlib Crop Geometry)
TARGET_WIDTH = 128
TARGET_HEIGHT = 64

# Temporal & Sequence Specifications
NUM_CLASSES = 11
NUM_FRAMES = 30
GRAYSCALE = False
CHANNELS = 1 if GRAYSCALE else 3

# DataLoader Configuration
BATCH_SIZE = 16
AUGMENT_TRAIN = False  # Kept False for baseline; configurable for training

# Temporally-Consistent Augmentation Parameters
ROTATION_FACTOR = 0.03
TRANSLATION_PAD = 4
BRIGHTNESS_DELTA = 0.08
CONTRAST_RANGE = (0.9, 1.1)

# Dataset Structure & Partition Targets (Speaker-Independent)
EXPECTED_SPEAKERS = [f"s{i}" for i in range(1, 31)]
EXPECTED_DIGITS = [f"d{i}" for i in range(11)]
EXPECTED_UTTERANCES = len(EXPECTED_SPEAKERS) * len(EXPECTED_DIGITS)  # 330
TRAIN_SPEAKERS_COUNT = 24
VAL_SPEAKERS_COUNT = 3
TEST_SPEAKERS_COUNT = 3

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

print(f"Project Root:        {PROJECT_ROOT.resolve()}")
print(f"Cropped Lip Dataset: {INPUT_CROPS_ROOT.resolve()}")
print(f"Results Directory:   {RESULTS_DIR.resolve()}")
print(f"Target Dimensions:   {TARGET_WIDTH}x{TARGET_HEIGHT} (Channels: {CHANNELS})")
print(f"Sequence Length (T): {NUM_FRAMES} frames")
print(f"Batch Size (B):      {BATCH_SIZE}")
print(f"Augment Train:       {AUGMENT_TRAIN}")
print(f"Expected Utterances: {EXPECTED_UTTERANCES} ({len(EXPECTED_SPEAKERS)} spk x {len(EXPECTED_DIGITS)} dig)")

Project Root:        /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading
Cropped Lip Dataset: /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/data/cropped_lips_dataset
Results Directory:   /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/results/dlib
Target Dimensions:   128x64 (Channels: 3)
Sequence Length (T): 30 frames
Batch Size (B):      16
Augment Train:       False
Expected Utterances: 330 (30 spk x 11 dig)


## 3. Dataset Discovery & Health Audit
We systematically discover the canonical 330 utterances from `data/cropped_lips_dataset/`.
- Frames are strictly sorted **numerically**, avoiding lexicographical misordering.
- We validate continuity, absence of sequence gaps, empty directories, and unreadable files.

In [3]:
# Section 3: Dataset Discovery & Frame Validation
def natural_sort_key(path):
    match = re.search(r"(\d+)", Path(path).stem)
    return int(match.group(1)) if match else 0

discovered_records = []
missing_utterances = []
invalid_utterances = []

for spk in EXPECTED_SPEAKERS:
    for dig in EXPECTED_DIGITS:
        sample_id = f"{spk}_{dig}"
        dir_path = INPUT_CROPS_ROOT / spk / dig

        if not dir_path.exists():
            missing_utterances.append({"sample_id": sample_id, "speaker": spk, "digit": dig, "reason": "Missing directory"})
            continue

        frames = sorted(
            [f for f in dir_path.glob("*.*") if f.suffix.lower() in VALID_EXTENSIONS],
            key=natural_sort_key
        )

        if len(frames) == 0:
            invalid_utterances.append({"sample_id": sample_id, "speaker": spk, "digit": dig, "reason": "Empty directory"})
            continue

        # Continuity validation
        indices = [natural_sort_key(f) for f in frames]
        expected = list(range(indices[0], indices[0] + len(frames)))
        if indices != expected:
            invalid_utterances.append({"sample_id": sample_id, "speaker": spk, "digit": dig, "reason": "Sequence gap"})

        discovered_records.append({
            "sample_id": sample_id,
            "speaker": spk,
            "digit": dig,
            "frame_directory": str(dir_path.resolve()),
            "num_frames": len(frames)
        })

manifest_df = pd.DataFrame(discovered_records)

print("=" * 60)
print("           DATASET DISCOVERY & AUDIT REPORT")
print("=" * 60)
print(f"Expected Utterances:   {EXPECTED_UTTERANCES}")
print(f"Discovered:            {len(manifest_df)}")
print(f"Missing:               {len(missing_utterances)}")
print(f"Invalid:               {len(invalid_utterances)}")
print(f"Total Lip Frames:      {manifest_df['num_frames'].sum()}")
print("=" * 60)
assert len(manifest_df) == EXPECTED_UTTERANCES

           DATASET DISCOVERY & AUDIT REPORT
Expected Utterances:   330
Discovered:            330
Missing:               0
Invalid:               0
Total Lip Frames:      13610


## 4. Class Mapping & Manifest
Deterministic mapping from digit strings (`d0`–`d10`) to integer labels (0–10).

In [ ]:
# Section 4: Deterministic Class Mapping
CLASS_TO_ID = {d: i for i, d in enumerate(EXPECTED_DIGITS)}
ID_TO_CLASS = {i: d for i, d in enumerate(EXPECTED_DIGITS)}

class_mapping_data = {
    "class_to_id": CLASS_TO_ID,
    "id_to_class": ID_TO_CLASS,
    "num_classes": NUM_CLASSES
}
class_mapping_path = RESULTS_DIR / "class_mapping.json"
with open(class_mapping_path, "w") as f:
    json.dump(class_mapping_data, f, indent=2)

print(f"Class mapping saved to: {class_mapping_path.resolve()}")
print("Mapping:", CLASS_TO_ID)

Class mapping saved to: /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/results/dlib/class_mapping.json
Mapping: {'d0': 0, 'd1': 1, 'd2': 2, 'd3': 3, 'd4': 4, 'd5': 5, 'd6': 6, 'd7': 7, 'd8': 8, 'd9': 9, 'd10': 10}


## 5. Speaker-Independent Split
We construct a **strict speaker-independent partition** using `SEED = 42`:
- **Train (24 speakers)**: 264 utterances
- **Validation (3 speakers)**: 33 utterances
- **Test (3 speakers)**: 33 utterances

Zero speaker overlap is verified to eliminate potential data leakage across splits.

In [5]:
# Section 5: Speaker Split & Leakage Verification
rng = random.Random(SEED)
shuffled_speakers = list(EXPECTED_SPEAKERS)
rng.shuffle(shuffled_speakers)

train_speakers = sorted(shuffled_speakers[:TRAIN_SPEAKERS_COUNT], key=lambda x: int(x[1:]))
val_speakers = sorted(shuffled_speakers[TRAIN_SPEAKERS_COUNT:TRAIN_SPEAKERS_COUNT + VAL_SPEAKERS_COUNT], key=lambda x: int(x[1:]))
test_speakers = sorted(shuffled_speakers[TRAIN_SPEAKERS_COUNT + VAL_SPEAKERS_COUNT:], key=lambda x: int(x[1:]))

speaker_to_split = {}
for s in train_speakers: speaker_to_split[s] = "train"
for s in val_speakers: speaker_to_split[s] = "val"
for s in test_speakers: speaker_to_split[s] = "test"

manifest_df["split"] = manifest_df["speaker"].map(speaker_to_split)

# Leakage Verification
set_train, set_val, set_test = set(train_speakers), set(val_speakers), set(test_speakers)
assert len(set_train & set_val) == 0, "Leakage between Train and Val!"
assert len(set_train & set_test) == 0, "Leakage between Train and Test!"
assert len(set_val & set_test) == 0, "Leakage between Val and Test!"

print("=" * 65)
print("DATA LEAKAGE CHECK: PASSED")
print("=" * 65)
print(f"Train Speakers ({len(train_speakers)}):      {train_speakers}")
print(f"Validation Speakers ({len(val_speakers)}): {val_speakers}")
print(f"Test Speakers ({len(test_speakers)}):       {test_speakers}")

# Save Speaker Split JSON
split_info = {
    "seed": SEED,
    "train_speakers": train_speakers,
    "val_speakers": val_speakers,
    "test_speakers": test_speakers,
    "train_utterances": int((manifest_df['split'] == 'train').sum()),
    "val_utterances": int((manifest_df['split'] == 'val').sum()),
    "test_utterances": int((manifest_df['split'] == 'test').sum()),
    "leakage_verified": True
}
split_json_path = RESULTS_DIR / "speaker_split.json"
with open(split_json_path, "w") as f:
    json.dump(split_info, f, indent=2)

# Save Manifest CSV
manifest_csv_path = RESULTS_DIR / "preprocessing_manifest.csv"
manifest_df.to_csv(manifest_csv_path, index=False)
print(f"Manifest CSV saved to:      {manifest_csv_path.resolve()}")
print(f"Speaker split JSON saved to: {split_json_path.resolve()}")
manifest_df.head(10)

DATA LEAKAGE CHECK: PASSED
Train Speakers (24):      ['s2', 's3', 's5', 's6', 's7', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's22', 's23', 's25', 's26', 's27', 's28', 's29', 's30']
Validation Speakers (3): ['s8', 's9', 's24']
Test Speakers (3):       ['s1', 's4', 's21']
Manifest CSV saved to:      /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/results/dlib/preprocessing_manifest.csv
Speaker split JSON saved to: /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/results/dlib/speaker_split.json


,sample_id,speaker,digit,frame_directory,num_frames,split
0,s1_d0,s1,d0,/Users/milinda/ghq/github.com/legion2004/tai-p...,40,test
1,s1_d1,s1,d1,/Users/milinda/ghq/github.com/legion2004/tai-p...,39,test
2,s1_d2,s1,d2,/Users/milinda/ghq/github.com/legion2004/tai-p...,40,test
3,s1_d3,s1,d3,/Users/milinda/ghq/github.com/legion2004/tai-p...,36,test
4,s1_d4,s1,d4,/Users/milinda/ghq/github.com/legion2004/tai-p...,48,test
5,s1_d5,s1,d5,/Users/milinda/ghq/github.com/legion2004/tai-p...,55,test
6,s1_d6,s1,d6,/Users/milinda/ghq/github.com/legion2004/tai-p...,35,test
7,s1_d7,s1,d7,/Users/milinda/ghq/github.com/legion2004/tai-p...,41,test
8,s1_d8,s1,d8,/Users/milinda/ghq/github.com/legion2004/tai-p...,42,test
9,s1_d9,s1,d9,/Users/milinda/ghq/github.com/legion2004/tai-p...,36,test


## 6. Deterministic Temporal Normalization
Every sample must produce exactly $T = 30$ frames:
- If $>30$ frames: **uniformly downsample** across the sequence.
- If $=30$ frames: identity mapping.
- If $<30$ frames: **deterministic nearest-neighbor interpolation/repetition**.
- Strictly deterministic; zero temporal jitter.

In [6]:
# Section 6: Temporal Sampling Strategy
def sample_frame_paths(frame_paths, target_frames=NUM_FRAMES):
    """
    Deterministically normalizes any sequence of frame paths to exactly target_frames (T=30).
    """
    n = len(frame_paths)
    if n == 0:
        raise ValueError("Cannot sample from empty frame_paths sequence.")
    indices = np.round(np.linspace(0, n - 1, target_frames)).astype(int)
    return [str(frame_paths[idx]) for idx in indices]

# Verify across multiple frame counts
test_20 = sample_frame_paths([f"f_{i}" for i in range(20)], 30)
test_30 = sample_frame_paths([f"f_{i}" for i in range(30)], 30)
test_45 = sample_frame_paths([f"f_{i}" for i in range(45)], 30)
assert len(test_20) == 30 and len(test_30) == 30 and len(test_45) == 30
print("[PASS] Temporal sampling produces exactly T=30 frames deterministically.")

[PASS] Temporal sampling produces exactly T=30 frames deterministically.


## 7. TensorFlow Native Frame Loading
Images are loaded directly into TensorFlow tensors using `tf.io.read_file` and `tf.image.decode_jpeg`, resized to $(96, 96)$, and normalized to $[0, 1]$ float32.

In [7]:
# Section 7: Native TensorFlow Frame Loading
def load_single_frame(path_tensor, target_h=TARGET_HEIGHT, target_w=TARGET_WIDTH, channels=CHANNELS):
    raw = tf.io.read_file(path_tensor)
    img = tf.image.decode_jpeg(raw, channels=channels)
    img = tf.image.resize(img, [target_h, target_w], method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32) / 255.0  # Normalize to [0, 1]
    img.set_shape([target_h, target_w, channels])
    return img

def load_video_frames(frame_paths_tensor, target_h=TARGET_HEIGHT, target_w=TARGET_WIDTH, channels=CHANNELS):
    frames = tf.map_fn(
        fn=lambda p: load_single_frame(p, target_h, target_w, channels),
        elems=frame_paths_tensor,
        fn_output_signature=tf.TensorSpec(shape=[target_h, target_w, channels], dtype=tf.float32)
    )
    frames.set_shape([NUM_FRAMES, target_h, target_w, channels])
    return frames

# Test loading one sample utterance
test_sample_paths = sample_frame_paths(
    sorted([f for f in (INPUT_CROPS_ROOT / 's1' / 'd0').glob('*.jpg')], key=natural_sort_key),
    NUM_FRAMES
)
sample_tensor = load_video_frames(tf.constant(test_sample_paths))
print(f"Loaded utterance tensor shape: {sample_tensor.shape}")
print(f"Tensor dtype:                 {sample_tensor.dtype}")
print(f"Pixel Range:                  [{tf.reduce_min(sample_tensor):.3f}, {tf.reduce_max(sample_tensor):.3f}]")
assert sample_tensor.shape == (NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS)

Loaded utterance tensor shape: (30, 64, 128, 3)
Tensor dtype:                 <dtype: 'float32'>
Pixel Range:                  [0.000, 0.827]


## 8. Temporally-Consistent Training Augmentation
Augmentation is applied strictly to training data when `AUGMENT_TRAIN = True` (disabled by default for baseline).
- **Temporally Consistent**: Parameters are sampled once per video sequence to avoid inter-frame flickering.
- Subtle brightness/contrast adjustment and small translation via symmetric padding & uniform crop.
- Horizontal flipping is disabled by default to preserve phonetic asymmetry.

In [8]:
# Section 8: Temporally-Consistent Video Augmentation
def augment_video(frames):
    # 1. Photometric jitter (uniform across all T frames)
    delta_b = tf.random.uniform([], -BRIGHTNESS_DELTA, BRIGHTNESS_DELTA)
    factor_c = tf.random.uniform([], CONTRAST_RANGE[0], CONTRAST_RANGE[1])
    frames = frames + delta_b
    frames = (frames - 0.5) * factor_c + 0.5
    frames = tf.clip_by_value(frames, 0.0, 1.0)

    # 2. Small Spatial Translation (uniform across all T frames)
    padded = tf.pad(frames, [[0, 0], [TRANSLATION_PAD, TRANSLATION_PAD], [TRANSLATION_PAD, TRANSLATION_PAD], [0, 0]], mode="SYMMETRIC")
    offset_y = tf.random.uniform([], 0, 2 * TRANSLATION_PAD + 1, dtype=tf.int32)
    offset_x = tf.random.uniform([], 0, 2 * TRANSLATION_PAD + 1, dtype=tf.int32)
    frames = padded[:, offset_y : offset_y + TARGET_HEIGHT, offset_x : offset_x + TARGET_WIDTH, :]
    frames.set_shape([NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS])
    return frames

aug_test = augment_video(sample_tensor)
print(f"Augmented tensor shape: {aug_test.shape}")
print(f"Augmented range:        [{tf.reduce_min(aug_test):.3f}, {tf.reduce_max(aug_test):.3f}]")

Augmented tensor shape: (30, 64, 128, 3)
Augmented range:        [0.009, 0.867]


## 9. High-Throughput tf.data Pipeline
We construct `train_dataset`, `val_dataset`, and `test_dataset` with in-memory caching to eliminate repeated disk I/O across training epochs.

In [9]:
# Section 9: tf.data Dataset Builder
def create_tf_dataset(df_subset, is_training=False, batch_size=BATCH_SIZE, augment=AUGMENT_TRAIN, shuffle=True):
    all_sampled_paths, all_labels = [], []
    for _, row in df_subset.iterrows():
        dir_p = Path(row["frame_directory"])
        fpaths = sorted(
            [f for f in dir_p.glob("*.*") if f.suffix.lower() in VALID_EXTENSIONS],
            key=natural_sort_key
        )
        all_sampled_paths.append(sample_frame_paths(fpaths, NUM_FRAMES))
        all_labels.append(CLASS_TO_ID[row["digit"]])

    dataset = tf.data.Dataset.from_tensor_slices((all_sampled_paths, all_labels))

    # Load and normalize frames
    dataset = dataset.map(
        lambda paths, lbl: (load_video_frames(paths, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS), tf.cast(lbl, tf.int32)),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # In-memory RAM caching for fast multi-epoch throughput
    dataset = dataset.cache()

    if is_training and shuffle:
        dataset = dataset.shuffle(buffer_size=len(df_subset), seed=SEED)

    if is_training and augment:
        dataset = dataset.map(
            lambda frames, lbl: (augment_video(frames), lbl),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_df = manifest_df[manifest_df["split"] == "train"].reset_index(drop=True)
val_df = manifest_df[manifest_df["split"] == "val"].reset_index(drop=True)
test_df = manifest_df[manifest_df["split"] == "test"].reset_index(drop=True)

train_dataset = create_tf_dataset(train_df, is_training=True, batch_size=BATCH_SIZE, augment=AUGMENT_TRAIN, shuffle=True)
val_dataset = create_tf_dataset(val_df, is_training=False, batch_size=BATCH_SIZE, augment=False, shuffle=False)
test_dataset = create_tf_dataset(test_df, is_training=False, batch_size=BATCH_SIZE, augment=False, shuffle=False)

print(f"train_dataset batches: {len(train_dataset)} (Batch Size: {BATCH_SIZE})")
print(f"val_dataset batches:   {len(val_dataset)}")
print(f"test_dataset batches:  {len(test_dataset)}")

train_dataset batches: 17 (Batch Size: 16)
val_dataset batches:   3
test_dataset batches:  3


## 10. Pipeline Validation & Batch Inspection
Complete sanity checks across dataset counts, class distributions, frame statistics, and batch tensor shapes.

In [ ]:
# Section 10: Pipeline Validation & Batch Inspection
print("=" * 70)
print("               COMPREHENSIVE PIPELINE VALIDATION")
print("=" * 70)
print("1. DATASET TOTALS:")
print(f"   Expected Utterances:   {EXPECTED_UTTERANCES}")
print(f"   Discovered:            {len(manifest_df)}")
print(f"   Missing:               {len(missing_utterances)}")
print(f"   Invalid:               {len(invalid_utterances)}")

print("\n2. SPLIT BREAKDOWN:")
print(f"   Train Utterances:      {len(train_df):>3} | Train Speakers:      {len(train_speakers):>2}")
print(f"   Val Utterances:        {len(val_df):>3} | Val Speakers:        {len(val_speakers):>2}")
print(f"   Test Utterances:       {len(test_df):>3} | Test Speakers:       {len(test_speakers):>2}")

print("\n3. ORIGINAL FRAME COUNT STATISTICS:")
fc = manifest_df["num_frames"]
print(f"   Min Frames:            {fc.min()}")
print(f"   Max Frames:            {fc.max()}")
print(f"   Mean Frames:           {fc.mean():.2f}")
print(f"   Median Frames:         {fc.median():.1f}")

print("\n4. CLASS DISTRIBUTION PER SPLIT:")
dist_df = pd.DataFrame({
    "Train": train_df["digit"].value_counts().sort_index(),
    "Val": val_df["digit"].value_counts().sort_index(),
    "Test": test_df["digit"].value_counts().sort_index(),
    "Total": manifest_df["digit"].value_counts().sort_index()
})
print(dist_df.to_string())

print("\n5. BATCH SHAPE VERIFICATION:")
x_batch, y_batch = next(iter(train_dataset))
print(f"   x_batch shape:         {x_batch.shape}")
print(f"   y_batch shape:         {y_batch.shape}")
print(f"   x_batch dtype:         {x_batch.dtype}")
print(f"   y_batch dtype:         {y_batch.dtype}")
print(f"   Pixel Range:           [{tf.reduce_min(x_batch):.3f}, {tf.reduce_max(x_batch):.3f}]")
print(f"   Sample Batch Labels:   {y_batch.numpy().tolist()}")

expected_batch_shape = (BATCH_SIZE, NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS)
assert x_batch.shape == expected_batch_shape
assert y_batch.shape == (BATCH_SIZE,)
print("\n   [PASS] Batch tensor shape matches exact expected dimensions!")
print("=" * 70)

               COMPREHENSIVE PIPELINE VALIDATION
1. DATASET TOTALS:
   Expected Utterances:   330
   Discovered:            330
   Missing:               0
   Invalid:               0

2. SPLIT BREAKDOWN:
   Train Utterances:      264 | Train Speakers:      24
   Val Utterances:         33 | Val Speakers:         3
   Test Utterances:        33 | Test Speakers:        3

3. ORIGINAL FRAME COUNT STATISTICS:
   Min Frames:            15
   Max Frames:            92
   Mean Frames:           41.24
   Median Frames:         40.0

4. CLASS DISTRIBUTION PER SPLIT:
       Train  Val  Test  Total
digit                         
d0        24    3     3     30
d1        24    3     3     30
d10       24    3     3     30
d2        24    3     3     30
d3        24    3     3     30
d4        24    3     3     30
d5        24    3     3     30
d6        24    3     3     30
d7        24    3     3     30
d8        24    3     3     30
d9        24    3     3     30

5. BATCH SHAPE VERIFICATION:
  

## 11. Save Preprocessing Configuration
We persist the full configuration and file pointers into `results/dlib/preprocessing_config.json`.

In [11]:
# Section 11: Configuration Persistence
preprocessing_config = {
    "seed": SEED,
    "num_classes": NUM_CLASSES,
    "num_frames": NUM_FRAMES,
    "image_width": TARGET_WIDTH,
    "image_height": TARGET_HEIGHT,
    "channels": CHANNELS,
    "grayscale": GRAYSCALE,
    "batch_size": BATCH_SIZE,
    "augmentation_enabled": AUGMENT_TRAIN,
    "split_information": {
        "train_speakers": train_speakers,
        "val_speakers": val_speakers,
        "test_speakers": test_speakers,
        "train_utterances": len(train_df),
        "val_utterances": len(val_df),
        "test_utterances": len(test_df),
        "total_utterances": len(manifest_df)
    },
    "artifacts": {
        "manifest_csv": str(manifest_csv_path.resolve()),
        "speaker_split_json": str(split_json_path.resolve()),
        "class_mapping_json": str(class_mapping_path.resolve())
    }
}

config_json_path = RESULTS_DIR / "preprocessing_config.json"
with open(config_json_path, "w") as f:
    json.dump(preprocessing_config, f, indent=2)
print(f"Preprocessing configuration saved successfully to:")
print(f"  {config_json_path.resolve()}")

Preprocessing configuration saved successfully to:
  /Users/milinda/ghq/github.com/legion2004/tai-phake-lip-reading/results/dlib/preprocessing_config.json


## 12. Model Architecture Interface Verification
We verify that the input batch `(B, 30, 96, 96, 3)` connects directly to a Keras model utilizing `TimeDistributed(CNN)` followed by a recurrent sequence aggregator.

In [12]:
# Section 12: TimeDistributed Model Interface Verification
input_layer = tf.keras.layers.Input(
    shape=(NUM_FRAMES, TARGET_HEIGHT, TARGET_WIDTH, CHANNELS),
    name="visual_sequence_input"
)

cnn_extractor = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), strides=(2, 2), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu")
], name="spatial_cnn")

temporal_features = tf.keras.layers.TimeDistributed(cnn_extractor, name="time_distributed_cnn")(input_layer)
recurrent_out = tf.keras.layers.GRU(64, name="temporal_gru")(temporal_features)
drop = tf.keras.layers.Dropout(0.3)(recurrent_out)
output_logits = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="digit_classifier")(drop)

verification_model = tf.keras.Model(inputs=input_layer, outputs=output_logits, name="vsr_verification_model")
pred_batch = verification_model(x_batch)

print("Keras Model Input Compatibility Verification:")
print(f"  Model Input Shape:  {verification_model.input_shape}")
print(f"  Model Output Shape: {verification_model.output_shape}")
print(f"  Batch Pass Shape:   {pred_batch.shape}")
assert pred_batch.shape == (BATCH_SIZE, NUM_CLASSES)
print("  [PASS] Pipeline is 100% compatible with tf.keras TimeDistributed architectures!")

Keras Model Input Compatibility Verification:
  Model Input Shape:  (None, 30, 64, 128, 3)
  Model Output Shape: (None, 11)
  Batch Pass Shape:   (16, 11)
  [PASS] Pipeline is 100% compatible with tf.keras TimeDistributed architectures!
